# Activity 5 – Special Data Types
**Course:** Advanced Programming – Week 6  
**Author:** Sebastian Diaz  

Uses McKinney (2017) textbook Chapter 14 datasets.
- **Exercise 1**: Identify data types in the Chapter 14 datasets
- **Exercise 2**: MovieLens dataset analysis (Section 14.2)

**MovieLens data download:** https://grouplens.org/datasets/movielens/1m/  
Extract into a `ml-1m/` folder containing `users.dat`, `movies.dat`, `ratings.dat`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, os

---
## Exercise 1 – Identify Data Types in Chapter 14 Datasets

### Time data (identify timezone)

In the **MovieLens** dataset (`ratings.dat`), each rating includes a Unix timestamp:
```
UserID::MovieID::Rating::Timestamp
1::1193::5::978300760
```
Unix timestamps are **UTC** (Coordinated Universal Time), also called epoch time (seconds since 1970-01-01 00:00:00 UTC). This is an important consideration: the MovieLens ratings were made by US-based users, so when displayed locally the timestamps should be converted to the relevant US timezone (e.g., US/Eastern, US/Pacific).

**Timezone:** UTC (source), requires conversion for local interpretation.

```python
# Converting Unix timestamp to UTC datetime
pd.to_datetime(978300760, unit='s', utc=True)
# → Timestamp('2000-12-31 09:12:40+0000', tz='UTC')

# Convert to US/Eastern:
pd.to_datetime(978300760, unit='s', utc=True).tz_convert('US/Eastern')
# → Timestamp('2000-12-31 04:12:40-0500', tz='US/Eastern')
```

---

### Numerical continuous data

In the **USDA Food Database** (`database.json`), the `value` field for each nutrient is **continuous numerical data** — it represents the measured quantity (e.g., milligrams of potassium per 100g of food) and can take any positive real value. Examples: `Protein: 25.18g`, `Potassium: 407mg`, `Water: 62.4g`. These can be used directly in arithmetic and statistical operations (mean, standard deviation, correlation).

Similarly, in the **Baby Names** dataset, the `births` column is a count (technically discrete, but for analytical purposes treated as continuous when aggregated or modelled over time).

---

### Nominal data

In the MovieLens `users.dat` file:
```
UserID::Gender::Age::Occupation::Zip-code
```
- `Gender` (M/F) is **dichotomous** (a subtype of nominal).
- `Zip-code` is **nominal** — it is a label that identifies a geographic area. The numbers have no mathematical meaning (zip code 10001 is not "more" or "less" than 20001 — they are simply identifiers). Sorting, averaging or subtracting zip codes is meaningless.

In the USDA database, `food_group` (e.g., 'Dairy and Egg Products', 'Legumes') is nominal.

---

### Coded categorical data (numerical representation of categories)

In the MovieLens `users.dat` file:
- `Occupation` is stored as an integer (0–20), but these numbers are codes for occupational categories (e.g., 0='other', 4='college/grad student', 17='scientist'). The integers have **no ordinal or numerical meaning** — occupation 17 is not 17 times occupation 1. This is categorical data that has been numerically coded for storage efficiency.
- `Age` is stored as one of seven integer codes (1, 18, 25, 35, 45, 50, 56) representing **age bands** — not the actual age. This is also coded categorical (ordinal) data. The code 56 means 'age 56+', not the number 56.

This is a common source of analytical errors — naive averaging of `Occupation` or `Age` codes produces nonsense.

---
## Exercise 2 – MovieLens Dataset Analysis

### Load the data

In [ ]:
# Download from https://grouplens.org/datasets/movielens/1m/
# Extract to ml-1m/ folder

DATA_DIR = 'ml-1m'

# Users
unames = ['user_id', 'gender', 'age', 'occupation', 'zip']
users  = pd.read_csv(os.path.join(DATA_DIR, 'users.dat'),
                     sep='::', engine='python', names=unames)

# Ratings
rnames  = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv(os.path.join(DATA_DIR, 'ratings.dat'),
                      sep='::', engine='python', names=rnames)

# Movies
mnames = ['movie_id', 'title', 'genres']
movies = pd.read_csv(os.path.join(DATA_DIR, 'movies.dat'),
                     sep='::', engine='python', names=mnames,
                     encoding='latin-1')

print("Users:",   users.shape,   '| Types:', users.dtypes.to_dict())
print("Ratings:", ratings.shape, '| Types:', ratings.dtypes.to_dict())
print("Movies:",  movies.shape,  '| Types:', movies.dtypes.to_dict())

In [ ]:
# --- Information Need ---
# Question: Do male and female users rate movies in the same Action and Romance
# genres differently? Is there a statistically meaningful gender gap in ratings
# for genre-specific films?
#
# Why: Recommendation systems trained on rating data without accounting for
# demographic differences may propagate gender-based preference bias.

# --- Step 1: Clean and prepare ---
# Convert timestamp to datetime (UTC)
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s', utc=True)

# Age is coded — replace with readable labels
age_map = {1: 'Under 18', 18: '18-24', 25: '25-34',
           35: '35-44', 45: '45-49', 50: '50-55', 56: '56+'}
users['age_band'] = users['age'].map(age_map)

# --- Step 2: Merge all three tables ---
data = ratings.merge(users, on='user_id').merge(movies, on='movie_id')
print("Merged shape:", data.shape)
print(data[['title','genres','gender','age_band','rating']].head(3))

In [ ]:
# --- Step 3: Analyse genre ratings by gender ---

# Explode genres (each film can have multiple; we want per-genre analysis)
data['genre_list'] = data['genres'].str.split('|')
data_exploded = data.explode('genre_list').rename(columns={'genre_list': 'genre'})

# Mean rating per genre × gender
genre_gender = (
    data_exploded.groupby(['genre', 'gender'])['rating']
                 .agg(['mean', 'count'])
                 .round(3)
                 .reset_index()
)

# Pivot to wide format
pivot = genre_gender.pivot(index='genre', columns='gender', values='mean')
pivot['gap (M-F)'] = (pivot['M'] - pivot['F']).round(3)
pivot = pivot.sort_values('gap (M-F)', ascending=False)

print("Mean rating by genre and gender:")
print(pivot.to_string())

In [ ]:
# --- Step 4: Visualise ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Rating gap by genre
pivot['gap (M-F)'].plot(kind='barh', ax=axes[0],
                        color=['#d73027' if v < 0 else '#4393c3'
                               for v in pivot['gap (M-F)']],
                        edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Gender Rating Gap per Genre\n(positive = men rate higher)', fontsize=11)
axes[0].set_xlabel('Mean Rating Difference (M − F)')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Plot 2: Absolute mean ratings M vs F for top-10 genres by volume
top_genres = data_exploded.groupby('genre')['rating'].count().nlargest(10).index
top_pivot  = pivot.loc[top_genres].sort_values('M')

x = np.arange(len(top_pivot))
w = 0.35
axes[1].barh(x - w/2, top_pivot['F'], w, label='Female', color='#e07070', edgecolor='white')
axes[1].barh(x + w/2, top_pivot['M'], w, label='Male',   color='#7090e0', edgecolor='white')
axes[1].set_yticks(x)
axes[1].set_yticklabels(top_pivot.index)
axes[1].set_title('Mean Rating by Gender — Top 10 Genres', fontsize=11)
axes[1].set_xlabel('Mean Rating')
axes[1].legend()
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('MovieLens 1M — Gender Rating Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('movielens_gender_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

### Decisions, considerations and assumptions

| Decision | Rationale |
|---|---|
| Used `unit='s', utc=True` for timestamps | Unix epoch is always UTC; local conversion would require user location data |
| Mapped `age` codes to readable bands | Prevents accidental arithmetic on age codes (e.g., mean(1,18,25) = 14.7, meaningless) |
| Used `explode()` on genres | Films have multiple genres; exploding allows per-genre analysis without double-counting workarounds |
| Excluded genres with very low counts | Small samples produce unreliable means |
| Defined gender gap as M−F | Positive = men rate higher; negative = women rate higher. This is a directional metric. |
| Left `zip` column unused | Zip code is nominal — no analysis was planned that required geographic grouping |
| Left `occupation` codes as integers | Not needed for this information need; would require a separate mapping dictionary if used |